# Compare the constant learning rate on the diagnostic folds

Question: does learning rate 0.0003 improve selected performance or reduce probability swings compared with 0.001?
Reuse the completed instrumented seed-42 reference. Train only the lower-rate candidate.
Keep original folds 01 and 04, dropout 0.2, context thirty, width sixteen, natural sampling, and budget 40/12 fixed.
Keep the same diagnostics, Adam, unweighted loss, clipping, and macro-F1 checkpoint selection.
The budget is two new fits and at most eighty epochs.

These eleven held-out people form a targeted diagnostic subset, not complete development validation.
Lower volatility can reflect slower fitting rather than better generalization. Do not select a global default from these folds alone.
NLL means negative log-likelihood; lower is better. Stage indices are 0 = Wake, 1 = Light, 2 = Deep, and 3 = REM.
Read docs/transformer-learning-rate.md before enabling preparation or training.

Completed results: docs/experiments/transformer-learning-rate-2026-11-12/README.md


## Locate the repository and declare execution switches

**What:** Find the checkout, configure logging, and separate setup, preparation, preflight, and training switches.
**Why:** Make every costly or environment-changing operation explicit. Report execution never starts optimization.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_learning_rate_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-learning-rate-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-learning-rate"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)


## Optional environment setup

**What:** Optionally clone this branch and install locked dependency groups with uv.
**Why:** Support another machine without environment-specific notebook branches. Reused features and reference artifacts must also be available.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the reference, fixed controls, and budget

**What:** Check reference artifact hashes, original fold assignments, model settings, and resolved training device.
**Why:** Isolate the learning-rate change and fail before training if the comparison inputs are unavailable or incompatible.
The reference contains the completed instrumented 0.001 fits; it is never retrained here.


In [ ]:
import json

import pandas as pd

from sleep_classifier.experiment_evidence import EpochEvidenceReport
from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.paired_architecture_report import PairedArchitectureReport

experiment = GroupedValidationExperiment(ROOT / "configs/transformer-learning-rate-grouped.yaml")
reference = GroupedValidationExperiment(ROOT / "configs/transformer-diagnostics-grouped.yaml")
plan = experiment.plan()
reference_summary = reference.output_dir / "summary.json"
if not reference_summary.exists():
    raise FileNotFoundError("The completed instrumented reference is required")
reference_saved = json.loads(reference_summary.read_text())
if reference_saved["plan"] != reference.plan():
    raise ValueError("The saved reference differs from its current fold plan")
reference_validation = PairedArchitectureReport(reference_summary, reference_summary).build()
reference_manifest = json.loads((reference.output_dir / "manifest.json").read_text())
resolved_device = str(TrainingConfig(experiment.model_settings).device())
if resolved_device != reference_manifest["device"]:
    raise ValueError("Candidate device differs from the measured reference device")
if plan["folds"] != reference_saved["plan"]["folds"]:
    raise ValueError("Candidate folds differ from the original diagnostic partitions")
if len(plan["folds"]) != 2 or plan["maximum_epochs"] != 80:
    raise ValueError("The candidate exceeds the agreed two-fit budget")
excluded = {"output_dir", "learning_rate"}
recipe = {key: value for key, value in experiment.model_settings.items() if key not in excluded}
reference_recipe = {key: value for key, value in reference.model_settings.items()
                    if key not in excluded}
if recipe != reference_recipe:
    raise ValueError("Another model, training, or evidence setting changed")
if reference.model_settings["learning_rate"] != 0.001:
    raise ValueError("The reference must use constant learning rate 0.001")
if experiment.model_settings["learning_rate"] != 0.0003:
    raise ValueError("The candidate must use constant learning rate 0.0003")
display(pd.DataFrame(plan["folds"]).set_index("name"))
display(pd.DataFrame([{"reference_lr": 0.001, "candidate_lr": 0.0003,
                       "new_fits": 2, "maximum_new_epochs": 80, "device": resolved_device}]))
logger.info("Reference verified; model, diagnostics, folds, and budget remain fixed")


## Prepare only the new candidate

**What:** Optionally copy development features and freeze the lower-rate fold configurations.
**Why:** Preserve original participant splits and inputs without overwriting or retraining the completed reference.
Enable preparation once for a fresh candidate output directory.


In [ ]:
if RUN_PREPARATION:
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config configs/transformer-learning-rate-grouped.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; inspect existing outputs before retrying")
else:
    logger.info("Candidate preparation skipped")


## Check both prepared candidates without optimization

**What:** Run each actual prepared fold through the no-gradient input and model check.
**Why:** Verify the resolved device, scaler inputs, tensor shapes, and model size before training.
The preflight does not create trained checkpoints or diagnostic epochs.


In [ ]:
if RUN_CHECK and (experiment.output_dir / "manifest.json").exists():
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        logger.info("Checking %s without optimization", fold["name"])
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Diagnostic preflight failed")
elif RUN_CHECK:
    logger.info("Preflight pending: prepare the two folds first")
else:
    logger.info("Preflight skipped")


## Train the lower-rate candidate when ready

**What:** Optionally fit seed-42 candidate models on the two selected folds.
**Why:** Test only the constant learning rate while retaining the existing evidence and checkpoint-selection protocol.
Keep preparation disabled once its manifest exists. Existing results are refused; automatic resume is unsupported.


In [ ]:
if RUN_TRAINING:
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config configs/transformer-learning-rate-grouped.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Candidate training failed; inspect retained evidence before retrying")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")


## Inspect reference and candidate status

**What:** Read available epoch evidence for both recipes without fitting a model.
**Why:** Distinguish a completed reference from prepared, partial, failed, or completed candidate results.
Missing measurements remain unavailable rather than becoming zero scores.


In [ ]:
reporters = {"reference": EpochEvidenceReport(reference.output_dir),
             "candidate": EpochEvidenceReport(experiment.output_dir)}
evidence_by_role = {role: reporter.build() for role, reporter in reporters.items()}
status_rows = [{"recipe": role, "study_status": evidence["status"], "fold": fold["fold"],
                "fold_status": fold["status"], "recorded_epochs": len(fold["epochs"])}
               for role, evidence in evidence_by_role.items() for fold in evidence["folds"]]
if status_rows:
    display(pd.DataFrame(status_rows).set_index(["recipe", "fold"]))
for role, evidence in evidence_by_role.items():
    if not evidence["folds"]:
        logger.info("%s is not prepared", role)


## Compare complete diagnostic curves

**What:** Plot every available epoch's losses, macro-F1, gradients, and actual learning rates for each recipe.
**Why:** Check whether smoother validation behavior accompanies comparable training progress rather than merely slower fitting.
Training evaluation NLL disables dropout at fixed end-of-epoch weights. Ordinary training loss uses changing weights and active dropout.


In [ ]:
import matplotlib.pyplot as plt

for role, reporter in reporters.items():
    if not any(fold["epochs"] for fold in evidence_by_role[role]["folds"]):
        logger.info("%s curves are pending", role)
        continue
    figure = reporter.plot_curves()
    figure.suptitle(f"{role}: constant learning rate "
                   f"{0.001 if role == 'reference' else 0.0003}")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    display(figure)
    plt.close(figure)


## Compare selected checkpoints only after completion

**What:** Validate both completed runs under the strict learning-rate comparison policy and display paired scores.
**Why:** Permit only the intended recipe change and compare the same target identities, participant folds, and training device.
Macro-F1 remains primary. Class precision, recall, and F1 expose trade-offs hidden by aggregate accuracy.


In [ ]:
comparison = None
candidate_summary = experiment.output_dir / "summary.json"
if candidate_summary.exists() and json.loads(candidate_summary.read_text())["status"] == "complete":
    comparison = PairedArchitectureReport(
        reference_summary, candidate_summary, comparison="learning_rate",
    ).build()
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    score_rows = [{"recipe": role, "accuracy": comparison[role]["accuracy"],
                   "pooled_macro_f1": comparison[role]["macro_f1"]}
                  for role in ("reference", "candidate")]
    display(pd.DataFrame(score_rows).set_index("recipe"))
    class_rows = [{"recipe": role, "stage": stage, **scores}
                  for role in ("reference", "candidate")
                  for stage, scores in comparison[role]["classes"].items()]
    display(pd.DataFrame(class_rows).set_index(["recipe", "stage"]))
    destination = experiment.output_dir / "paired-comparison.json"
    destination.write_text(json.dumps(comparison, indent=2) + "\n")
    logger.info("Paired subset comparison saved: %s", destination)
else:
    logger.info("The candidate is incomplete; no paired performance conclusion is available")


## Inspect selected NLL, fitting progress, clipping, and loss swings

**What:** Summarize selected-checkpoint probability loss and the largest upward validation-NLL step for each available fit.
**Why:** Separate improved selected performance from a quieter trajectory that simply learned less within forty epochs.
NLL is secondary; its minimum does not replace the macro-F1 checkpoint rule. Unequal completed lengths limit maximum-jump comparisons.


In [ ]:
diagnostic_rows = []
for role, evidence in evidence_by_role.items():
    for fold in evidence["folds"]:
        epochs = fold["epochs"]
        if not epochs:
            continue
        selected = next((epoch for epoch in epochs if epoch["epoch"] == fold["best_epoch"]), None)
        jumps = [{"after_epoch": after["epoch"],
                  "delta": after["validation_nll"] - before["validation_nll"]}
                 for before, after in zip(epochs, epochs[1:])]
        largest = max(jumps, key=lambda row: row["delta"]) if jumps else None
        diagnostic_rows.append({
            "recipe": role, "fold": fold["fold"], "completed_epochs": len(epochs),
            "selected_epoch": fold["best_epoch"],
            "selected_validation_nll": selected["validation_nll"] if selected else None,
            "selected_training_evaluation_nll": selected["training_evaluation_nll"]
            if selected else None,
            "selected_clipped_step_fraction": selected["clipped_step_fraction"]
            if selected else None,
            "final_training_evaluation_nll": epochs[-1]["training_evaluation_nll"],
            "largest_upward_nll_step": max(0.0, largest["delta"]) if largest else None,
            "largest_upward_step_epoch": largest["after_epoch"]
            if largest and largest["delta"] > 0 else None,
        })
if diagnostic_rows:
    display(pd.DataFrame(diagnostic_rows).set_index(["recipe", "fold"]))
else:
    logger.info("Diagnostic scores are pending")


## Save portable local evidence reports

**What:** Regenerate each recipe's local evidence summary and any available figures through the CLI.
**Why:** Keep inspection reproducible outside Jupyter and retain partial-run evidence after a failure.
The inspector logs an unprepared path without creating it or blocking later preparation.


In [ ]:
%cd "{ROOT}"
for role, directory in (("reference", reference.output_dir), ("candidate", experiment.output_dir)):
    logger.info("Saving %s evidence report", role)
    !uv run --locked --group gru --group notebook python -m sleep_classifier.experiment_evidence \
        --output-dir "{directory}" --report
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Evidence inspection failed")


## Export the executed HTML report

**What:** Execute report cells in a fresh kernel and save one standalone HTML report.
**Why:** Share the current evidence while keeping source notebooks output-free and skipping preparation, training, setup, and recursive export.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/21_transformer_learning_rate.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Learning-rate notebook export failed")
